# Импортирование библиотек

In [1]:
import torch.nn.functional as F
import torch
from torch import Tensor
from transformers import AutoTokenizer, AutoModel
import datasets
from tqdm import tqdm
import requests
import json

/mnt/cs/home/tirskikh/miniconda3/envs/pl_template/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Предобработка данных

In [2]:
raw_dataset = datasets.load_dataset('json',data_files="./data/medotvet.json")
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['url', 'question', 'answers'],
        num_rows: 4318
    })
})

In [3]:
raw_dataset['train'][0]['question']

{'user_name': 'Enihka 23 года',
 'title': 'Вопрос №1000',
 'text': 'скажите  поставили диагноз киста правого яичника  увеличенных размеров , отправляют на операцию очень боюсь лечила всякими методами не чего не помогает ((( можно ли без вмешательства хирургов  как это удалить??'}

In [4]:
raw_dataset['train'][0]['answers']

[{'doctor_name': 'Ромаш А. В.',
  'doctor_spec': 'Эндокринолог г. Краснодар',
  'answer_text': 'Доброго вечера.Опишите заключение УЗИ- размеры яичников и размеры образования, при больших образованиях, действительно, единственным методом является удаление, в идеале- лапароскопическим доступом.При хорошей гистологии, необходимо начать прием оральных контрацептивов, с лечебной целью'}]

In [5]:
limit = 100
ds_dict = {'candidate': [], 'query': []}
query_2_id = {}
corpus = set()

for row in raw_dataset['train']:
    candidates = set([d['answer_text'] for d in row['answers']])
    query = f"Пользователь: {row['question']['user_name']} Запрос: {row['question']['text']}"
    
    if query not in query_2_id:
        ds_dict['candidate'].append(candidates)
        ds_dict['query'].append(query)
        query_2_id[query] = len(query_2_id)
    else:
        q_id = query_2_id[query]
        ds_dict['candidate'][q_id].update(candidates)

    corpus.update(candidates)
    
    if len(query_2_id) == limit:
        break

ds = datasets.Dataset.from_dict(ds_dict)
corpus_list = [c for c in list(corpus) if len(c)>0]
corpus = datasets.Dataset.from_dict({'id':list(range(len(corpus_list))),'text':corpus_list})
cand_2_id = {c:i for i,c in enumerate(corpus_list)}
queries = ds['query']

# Создание сложно-негативных примеров

<div>
<img src="./images/image.png" width="300"/>
</div>

In [6]:
def average_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

def encode(model, tokenizer, prefix, input_texts, batch_size = 2):
    # Tokenize the input texts
    input_texts = [f"{prefix}: {text}" for text in input_texts]
    
    embeddings_matrix = None
    
    for i in tqdm(range(0,len(input_texts),batch_size)):
        batch_dict = tokenizer(input_texts[i:i+batch_size], max_length=128, padding=True, truncation=True, return_tensors='pt').to('cuda:0')
        outputs = model(**batch_dict)
        embeddings = average_pool(outputs.last_hidden_state, batch_dict['attention_mask'])
        
        if embeddings_matrix is not None:
            embeddings_matrix = torch.concat((embeddings_matrix, embeddings), dim=0)
        else:
            embeddings_matrix = embeddings
    
    return embeddings_matrix

In [7]:
# загружаем модель для формирования сложно-негативных примеров
tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-small')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-small')
model.to('cuda:0')

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(250037, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=Fals

In [8]:
# получаем эмбеддинги запросов и кандидатов
query_embd = encode(model,tokenizer,'query',queries)
candidate_embd = encode(model,tokenizer,'passage',corpus)

scores = query_embd @ candidate_embd.T

100%|██████████| 80/80 [00:01<00:00, 47.66it/s]


In [9]:
ds = ds.map(lambda x, i:
        {'positives':{'doc_id':[cand_2_id[c] for c in x['candidate']],
                      'score':[scores[i][cand_2_id[c]] for c in x['candidate']]}},
        remove_columns='candidate',
        with_indices=True)

ds = ds.map(lambda x, i:
        {'negatives':{'doc_id':[cand_id for cand_id in list(cand_2_id.values()) if cand_id not in x['positives']['doc_id']],
                      'score':[scores[i][cand_id] for cand_id in list(cand_2_id.values()) if cand_id not in x['positives']['doc_id']]}},
        with_indices=True)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map: 100%|██████████| 100/100 [00:01<00:00, 80.55 examples/s]


# Фильтрация по скорам

<div>
<img src="./images/image-1.png" width="400"/>
</div>

In [10]:
import numpy as np 

def get_hard_negatives(x, corpus, alpha = 0.95, top_k=15, type='perc', pos_score='mean+std'):
    pos_scores = np.array(x['positives']['score'])
    
    if pos_score == 'min':
        pos_score = np.min(pos_scores)
    elif pos_score == 'max':
        pos_score = np.max(pos_scores)
    elif pos_score == 'mean':
        pos_score = np.mean(pos_scores)
    elif pos_score == 'mean+std':
        pos_score = np.mean(pos_scores) + np.std(pos_scores)
    
    neg_scores = np.array(x['negatives']['score'])
    scores_mask = np.ones_like(neg_scores, dtype=bool)
    sorted_scores_idx = np.argsort(-neg_scores)
    sorted_scores = neg_scores[sorted_scores_idx]
    sorted_idx = np.array(x['negatives']['doc_id'])[sorted_scores_idx]
    
    if type == 'abs':
        scores_mask[sorted_scores > alpha] = 0
    if type == 'margin':
        scores_mask[sorted_scores > pos_score-alpha] = 0
    if type == 'perc':
        scores_mask[sorted_scores > pos_score*alpha] = 0
    
    top_indices = sorted_idx[scores_mask].tolist()
    
    hard_negatives = []
    hard_negatives_set = set()
    for i in top_indices:
        text = corpus[i]['text']
        if text not in hard_negatives_set:
            hard_negatives.append(text)
            hard_negatives_set.add(text)
            
            if len(hard_negatives) == top_k:
                break
    
    return {'hard_negative_samples':hard_negatives}

In [11]:
ds = ds.map(get_hard_negatives, fn_kwargs={
                'corpus':corpus,
                "alpha":1,
                "top_k":10,
                "type":'naive',
                "pos_score": 'mean'
            })

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map: 100%|██████████| 100/100 [00:00<00:00, 481.00 examples/s]


In [12]:
print('query:', ds[1]['query'])
ds[1]['hard_negative_samples']

query: Пользователь: Вика 16 лет Запрос: Здравствуйте. У меня задержка месячных на 6 дней, 3 недели назад был незащищенный  половой акт, но он был прерван. Очень волнуюсь. Могу ли я быть беременна? Делала 3 теста, все показывали 1 полоску.


['Добрый день. Это может быть просто задержка. Или беременность. Сдайте анализ на ХГЧ, он достовернее тестов в плане беременности. И посетите гинеколога для осмотра. Спасибо.',
 'Доброго вечера. Сдайте кровь на пролактин ( за 3 дня исключите стресс, спорт , секс, раздражение молочных желез,утром натощак до 9.00),сделайте УЗИ молочных желез. Не переживайте.',
 'Здравствуйте, Елена! Вам нужно проконсультироваться у инфекциониста. Возможно, Вам предложат люмбальную пункцию для уточнения диагноза.',
 'Здравствуйте! Вам нужно обратиться к мануальному терапевту.',
 'Добрый день. Обратитесь к урологу для осмотра и обследования. Только после этого можно будет поставить диагноз и назначить лечение.',
 'Здравствуйте! Вам нужно показать ребенка детскому неврологу, чтобы получить ответ на свой вопрос.',
 'Половой покой минимум 14 дней.',
 'Добрый день. Вам срочно нужно сделать УЗИ почек, м/пузыря, простаты, ост мочи и попасть на консультацию к урологу.',
 'Здравствуйте! Вероятнее, причина не в сам

In [13]:
ds.save_to_disk('./data/naive_neg/dataset')
corpus.save_to_disk('./data/naive_neg/corpus')

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 159/159 [00:00<00:00, 1482.17 examples/s]


# Фильтрация с помощью LLM

<div>
<img src="./images/image-2.png" width="500"/>
</div>

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the model and tokenizer
model_name = "vikhrmodels/Vikhr-Qwen-2.5-1.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name)

ds = datasets.load_from_disk('./data/naive_neg/dataset')
corpus = datasets.load_from_disk('./data/naive_neg/corpus')

In [24]:
prompt_template = '''
Определи, является ли следующий документ релевантным запросу. 
Напиши “релевантный” или “не релевантный” в зависимости от ответа. 
Документ является релевантным если он отвечает на запрос или если он содержит ту же информацию что и релевантный документ.
Не повторяй содержание запроса или документа. Не давай никаких пояснений.

Запрос: {query}

Документ: {gt}
Результат: Релевантный

Документ: {hn}
Результат: 
'''

In [25]:
def map_llm_cutoff(row, corpus, patience = 5, max_attempts = 3, step=3, is_print=False):
    query = row['query']
    gt = corpus[row['positives']['doc_id'][0]]['text']
    
    negative_count = 0
    positive_ids = []
    last_ids = []
    errors = []
    found = True
    
    if is_print:
        print('query:', query)
        print('gt:', gt)
    
    for i, hn in enumerate(row['hard_negative_samples']):
        if i % step != 0 and not found:
            continue
        prompt = prompt_template.format(query=query, gt=gt, hn=hn)
        messages = [
            {"role": "system", "content": "Вы — Vikhr, ИИ помощник, созданный компанией Vikhr models для предоставления полезной, честной и безопасной информации."},
            {"role": "user", "content": prompt},
        ]
        
        attempts = 0
        
        while attempts < max_attempts:
            # Tokenize and generate text
            input_ids = tokenizer.apply_chat_template(messages, truncation=True, add_generation_prompt=True, return_tensors="pt")
            input_ids = input_ids.to('cuda')
            output = model.generate(
                input_ids,
                # max_length=1512,
                max_new_tokens=256,
                temperature=0.3,
                num_return_sequences=1,
            )

            # Decode and print result
            generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
            text = generated_text.split('assistant')[-1].strip().lower()
            if text in ['релевантный', 'не релевантный']:
                break
        else:
            errors.append((i, text))
            continue
        
        if text == 'не релевантный':
            negative_count += 1
            found = True
            if negative_count == patience:
                last_ids.append(i - patience)
                break
        elif text == 'релевантный':
            found = False
            negative_count = 0
            positive_ids.append(i)
        
        if is_print:
            print(i, text, negative_count, hn)
        
    return {'positive_ids':positive_ids, 'last_ids':last_ids, 'errors': errors}

In [26]:
map_llm_cutoff(ds[1], corpus, patience=5, step=3, max_attempts=3, is_print=True)

query: Пользователь: Вика 16 лет Запрос: Здравствуйте. У меня задержка месячных на 6 дней, 3 недели назад был незащищенный  половой акт, но он был прерван. Очень волнуюсь. Могу ли я быть беременна? Делала 3 теста, все показывали 1 полоску.
gt: Добрый вечер! Виктория, прерванный половой акт, не является способом контрацепции, все зависит от того куда кончил парень и не забывайте, если у парня обильная собственная смазка, в ней содержится небольшой % сперматозоидов! Предохраняйтесь презервативами!
0 релевантный 0 Добрый день. Это может быть просто задержка. Или беременность. Сдайте анализ на ХГЧ, он достовернее тестов в плане беременности. И посетите гинеколога для осмотра. Спасибо.
3 не релевантный 1 Здравствуйте! Вам нужно обратиться к мануальному терапевту.
4 не релевантный 2 Добрый день. Обратитесь к урологу для осмотра и обследования. Только после этого можно будет поставить диагноз и назначить лечение.
5 не релевантный 3 Здравствуйте! Вам нужно показать ребенка детскому неврологу, 

{'positive_ids': [0], 'last_ids': [2], 'errors': []}

# Метрики и результаты обучения

<div>
<img src="./images/image-3.png" width="600"/>
</div>